In [6]:
import json
import re
import cv2
import pytesseract
import shutil
import fitz

import numpy as np
import pandas as pd

from pathlib import Path
from typing import Optional
from pdf2image import convert_from_path
from PIL import Image
from img2table.document import Image as Img2TableImage
from img2table.ocr import TesseractOCR

In [8]:
def _preprocess_for_ocr(pil_img: Image.Image) -> Image.Image:
    img = np.array(pil_img.convert("RGB"))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    denoised = cv2.fastNlMeansDenoising(gray, h=20)
    bw = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return Image.fromarray(bw)

def _resolver_tesseract_cmd(tesseract_cmd: Optional[str] = None) -> str:
    if tesseract_cmd and Path(tesseract_cmd).exists():
        return tesseract_cmd

    candidato_path = shutil.which("tesseract")
    if candidato_path:
        return candidato_path

    candidatos = [
        r"C:\Program Files\Tesseract-OCR\tesseract.exe",
        r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"]
    
    for c in candidatos:
        if Path(c).exists():
            return c
    
    raise RuntimeError("No se pudo encontrar el ejecutable de Tesseract. Por favor, instálalo y asegúrate de que esté en tu PATH o proporciona la ruta correcta.")


def _pdf_a_imagenes(
    ruta_pdf: str, dpi: int = 300, poppler_path: Optional[str] = None,
    usar_fallback_pymupdf: bool = True) -> list[Image.Image]:
    try:
        return convert_from_path(ruta_pdf, dpi=dpi, poppler_path=poppler_path)
    except Exception as e_plopper:
        if not usar_fallback_pymupdf:
            raise e_plopper
    try:
        paginas = []
        doc = fitz.open(ruta_pdf)
        for pagina in doc:
            pix = pagina.get_pixmap(dpi=dpi)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            paginas.append(img)
    except Exception as e_fitz:
        raise RuntimeError(f"No se pudo convertir el PDF a imágenes usando poppler ni PyMuPDF. Errores:\nPoppler: {e_plopper}\nPyMuPDF: {e_fitz}")
    finally:
        doc.close()
    return paginas

def extraer_codigos_pdf_a_dataframe(
    ruta_pdf: str, idioma: str = "spa", dpi: int = 300,
    poppler_path: Optional[str] = None, tesseract_cmd: Optional[str] = None,
    regex_codigo: str = r"\b\d{9,}\b",) -> tuple[str, pd.DataFrame]:

    pytesseract.pytesseract.tesseract_cmd = _resolver_tesseract_cmd(tesseract_cmd)
    paginas = _pdf_a_imagenes(ruta_pdf, dpi=dpi, poppler_path=poppler_path)
    texto_partes = []
    registros = []

    for pagina_n, img in enumerate(paginas, start=1):
        procesada = _preprocess_for_ocr(img)
        texto = pytesseract.image_to_string(procesada, lang=idioma, config="--psm 6")
        texto_partes.append(f"\n--- Página {pagina_n} ---\n{texto}")

        codigos_encontrados = re.findall(regex_codigo, texto)
        for codigo in codigos_encontrados:
            registros.append({"pagina": pagina_n, "codigo": codigo})
    
    df_codigos = pd.DataFrame(registros).drop_duplicates().reset_index(drop=True)
    texto_total = "".join(texto_partes)
    return texto_total, df_codigos


In [9]:
ruta = "ruta/al/"  # Cambia esto a la ruta de tu archivo PDF
archivo = "archivo.pdf" # Cambia esto al nombre de tu archivo PDF
ruta_pdf = ruta + archivo

In [10]:
texto_ocr, df_codigos = extraer_codigos_pdf_a_dataframe(ruta_pdf, idioma="spa", dpi=300, poppler_path=None, tesseract_cmd=None, regex_codigo=r"\b\d{9,}\b")

print(f"Codigos encontrados: '{len(df_codigos)}'")
display(df_codigos.head(20))

df_codigos.to_csv("codigos_encontrados.csv", index=False, encoding="utf-8-sig")

RuntimeError: No se pudo encontrar el ejecutable de Tesseract. Por favor, instálalo y asegúrate de que esté en tu PATH o proporciona la ruta correcta.